# Final Model Pipeline
Frozen staged run: Core20 benchmarks, LightGBM breadth, cloud models, fixed ensemble, and robustness models. Completed annual refits and diagnostics are reused after interruption.

In [ ]:
import os, sys, importlib
from dataclasses import replace
from pathlib import Path

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/FDS')
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = DRIVE_PROJECT_DIR
if not (PROJECT_DIR / 'src').is_dir(): raise FileNotFoundError(PROJECT_DIR / 'src')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR)) if str(PROJECT_DIR) not in sys.path else None
for name in tuple(sys.modules):
    if name == 'src' or name.startswith('src.'): del sys.modules[name]
importlib.invalidate_caches()
import src
print('Project:', PROJECT_DIR)
print('Source:', Path(src.__file__).resolve())

In [ ]:
from src.config import (
    ExperimentConfig, UniverseConfig, FINAL_MODEL_ROSTER,
    STAGE_A_MODELS, STAGE_B_MODELS, STAGE_C_MODELS, STAGE_E_MODELS,
)
from src.runner import ExperimentRunner

BASE_CONFIG = ExperimentConfig(
    experiment_id='final_models_v1', project_dir=PROJECT_DIR,
    data_path=PROJECT_DIR / 'jkp_USA_100chars_1980_2024.parquet',
    output_dir=PROJECT_DIR / 'model_runs', selected_models=FINAL_MODEL_ROSTER,
    seed=42, use_gpu=True, universe=UniverseConfig(security_id_col='id'),
)
BASE_CONFIG.validate()
prepared_data = ExperimentRunner(BASE_CONFIG).prepare_data()

def run_stage(model_ids):
    config = replace(BASE_CONFIG, selected_models=tuple(model_ids))
    return ExperimentRunner(config).run(prepared_data=prepared_data)

def table_for(comparison, model_ids):
    order = {model: i for i, model in enumerate(model_ids)}
    return (comparison.loc[comparison.model_id.isin(model_ids)]
            .assign(_order=lambda x: x.model_id.map(order))
            .sort_values('_order').drop(columns='_order').reset_index(drop=True))

## Stage A — Core20 benchmark approaches

In [ ]:
comparison = run_stage(STAGE_A_MODELS)
stage_a_table = table_for(comparison, STAGE_A_MODELS)
stage_a_table

## Stage B — LightGBM characteristic breadth

In [ ]:
comparison = run_stage(STAGE_B_MODELS)
stage_b_table = table_for(comparison, STAGE_B_MODELS)
stage_b_table

In [ ]:
import matplotlib.pyplot as plt
breadth = stage_b_table.copy()
breadth['n_characteristics'] = breadth.model_id.str.extract(r'_(\d+)').astype(int)
for metric, label in [('pooled_oos_r2','Pooled OOS R²'), ('mean_monthly_rank_ic','Mean monthly Rank IC')]:
    ax = breadth.plot(x='n_characteristics', y=metric, marker='o', legend=False, figsize=(7,4))
    ax.set(xlabel='Number of characteristics', ylabel=label, title=f'LightGBM: {label}')
    ax.grid(alpha=.3); plt.show()

## Stage C — Static and dynamic cloud models

In [ ]:
comparison = run_stage(STAGE_C_MODELS)
print('Stage C complete.')

## Stage D — Fixed equal-weight ensemble

In [ ]:
from src.ensemble import ENSEMBLE_ID, build_fixed_fifty_fifty
ensemble_metrics = build_fixed_fifty_fifty(BASE_CONFIG)
ensemble_metrics

## Stage E — Robustness and matched comparison models
- **E1 neural depth:** NN3_20 and NN4_20, compared with NN2_20.
- **E2 temporal LightGBM:** LGBM_40_LAG1 and LGBM_40_LAG12, compared with LGBM_40.
- **E3 matched independent-stock network:** MLP_40, compared with DEEPSET_40.
- **E4 lagged cloud model:** DEEPSET_40_LAG1, between static and fully dynamic DeepSets.

In [ ]:
comparison = run_stage(STAGE_E_MODELS)
print('Stage E complete.')

## Focused cloud and ensemble comparison

In [ ]:
from src.model_comparison import build_model_comparison_table
comparison = build_model_comparison_table(BASE_CONFIG.run_dir)
CLOUD_COMPARISON = (
 'LGBM_40','LGBM_40_LAG1','LGBM_40_LAG12','MLP_40',
 'DEEPSET_40','DEEPSET_40_LAG1','DEEPSET_40_DYNAMIC',ENSEMBLE_ID,
)
cloud_table = table_for(comparison, CLOUD_COMPARISON)
cloud_table

## Neural-depth robustness

In [ ]:
NEURAL_DEPTH = ('NN2_20','NN3_20','NN4_20')
neural_depth_table = table_for(comparison, NEURAL_DEPTH)
neural_depth_table

## Final comparison — 16 standalone models plus fixed ensemble

In [ ]:
FINAL_ORDER = (*FINAL_MODEL_ROSTER, ENSEMBLE_ID)
final_comparison = table_for(comparison, FINAL_ORDER)
if len(final_comparison) != 17: raise RuntimeError(f'Expected 17 final results, found {len(final_comparison)}')
final_comparison

## Final training-artifact audit

In [ ]:
from src.post_train_audit import run_post_train_audit
training_audit = run_post_train_audit(BASE_CONFIG, standalone_model_ids=FINAL_MODEL_ROSTER, chosen_model_id=ENSEMBLE_ID, require_chosen_analysis=False)
failed = training_audit.loc[~training_audit.passed]
if not failed.empty:
    display(failed)
    raise RuntimeError(f'Training-artifact audit failed {len(failed)} checks')
print(f'TRAINING-ARTIFACT AUDIT PASS: {len(training_audit)} checks')